# 10 — Descarga de corridas → Drive

Resuelve los 19 accessions de `data/organismos.tsv` a corridas contra la ENA y
baja los `.sra` a `tesis/80_sra/`.

**Las sesiones de Colab se mueren, y eso es lo normal, no la excepción.** Este
notebook está hecho para eso: el estado del trabajo es qué archivos existen en
Drive, así que re-ejecutarlo retoma donde quedó. Usá `LIMITE` para que cada
sesión haga una tanda y termine.

Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]); 
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

In [ ]:
import shutil as _sh

SRA = DRIVE / '80_sra'; SRA.mkdir(parents=True, exist_ok=True)
STAGING = pathlib.Path('/content/sra_staging'); STAGING.mkdir(exist_ok=True)
MANIFIESTOS = DRIVE / '00_manifiestos'; MANIFIESTOS.mkdir(parents=True, exist_ok=True)
MANIFIESTO_DRIVE = MANIFIESTOS / 'srr_manifest.tsv'

# El ledger va a DRIVE, no adentro del clon. Dos motivos: el clon se resetea en
# cada corrida, asi que ahi adentro no sobrevive nada; y una sesion de Colab que
# se muere no puede llevarse los md5 con ella, que es exactamente lo que paso
# una vez y obligo a recalcularlos.
LEDGER_DRIVE = MANIFIESTOS / 'sra_md5.tsv'

env = dict(os.environ,
           SRA_DEST=str(SRA),
           SRA_STAGING=str(STAGING),
           SRA_LEDGER=str(LEDGER_DRIVE),
           MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'))


def _filas(p):
    return len(p.read_text().strip().split('\n')) - 1 if p.exists() else -1


# Sembrar el ledger de Drive desde la copia del repo la primera vez, y cada vez
# que el repo tenga mas filas porque commiteamos algo. Sin esto, el modo 'ledger'
# creeria que no hay nada registrado y recalcularia el md5 de las 416 corridas,
# que son ~190 GB de lectura sobre el mount de Drive.
_repo_led = CLON / 'data' / 'sra_md5.tsv'
if _filas(_repo_led) > _filas(LEDGER_DRIVE):
    _sh.copy(_repo_led, LEDGER_DRIVE)
    print('ledger sembrado desde el repo:', _filas(LEDGER_DRIVE), 'filas')

print('destino :', SRA)
print('staging :', STAGING, f'({_sh.disk_usage("/content").free/1e9:.0f} GB libres)')
print('ledger  :', LEDGER_DRIVE, f'({_filas(LEDGER_DRIVE)} filas)')


## 1. Manifiesto

Se genera con `scripts/fetch_runs.sh manifest`, que consulta la ENA y filtra a
datos de RNA: `library_source = TRANSCRIPTOMIC` —el filtro duro, porque varios
BioProjects mezclan corridas GENOMIC— y `SINGLE` para RNA-Seq, porque el PAIRED
de un proyecto de RNA-Seq no es sRNA-seq.

Se guarda una copia en Drive: el clon es efímero y el manifiesto define el
trabajo pendiente.

In [ ]:
import subprocess, shutil as _sh
if MANIFIESTO_DRIVE.exists():
    print('ya hay manifiesto en Drive; lo reuso.')
    print('Para regenerarlo, borralo primero.')
    _sh.copy(MANIFIESTO_DRIVE, CLON / 'data' / 'srr_manifest.tsv')
else:
    r = subprocess.run(['./scripts/fetch_runs.sh', 'manifest'], cwd=CLON,
                       capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    MANIFIESTO_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    _sh.copy(CLON / 'data' / 'srr_manifest.tsv', MANIFIESTO_DRIVE)
    print('copia guardada en', MANIFIESTO_DRIVE)

## 1b. Buscar un BioProject de reemplazo

Para cuando un duplicado resulta no ser sRNA-seq. Es el caso de `sclsc`: su
`PRJNA985401` es RNA-Seq PAIRED y cayó entero en el filtro, así que es **el
único de los 9 sin validación independiente** — y sin duplicado no hay con qué
confirmar un candidato priorizado.

Consulta la ENA con **el mismo filtro** que arma el manifiesto (`pasa_filtro()`,
compartida con el modo `manifest`), agrupa por BioProject y marca los que ya
están en `data/organismos.tsv`.

Un candidato válido tiene que decir `-` en la última columna: un proyecto que ya
está en la spec no valida nada de forma independiente.


In [ ]:
ESPECIES = ['Sclerotinia sclerotiorum']

for _esp in ESPECIES:
    cmd = ['./scripts/fetch_runs.sh', 'buscar', _esp]
    print('$ ' + ' '.join(cmd))
    p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    p.wait()
    print()


## 1c. ¿Estos datos son sRNA-seq de verdad?

`avg_len` y la etiqueta de estrategia de la ENA **no alcanzan**. `SRR23277331`
decía `miRNA-Seq` y no tenía un solo read con adaptador en 40 000: es mRNA.

Lo que decide es **dónde empieza el adaptador 3'**. Si el inserto pica en
18-30 nt, es sRNA-seq; si casi no hay adaptador, el inserto es más largo que el
read, que es lo que se espera de mRNA. La lista incluye `AGATCGGAAGAGC`, el
universal de Illumina, así que un inserto corto se ve **con cualquier kit**.

**`TODOS = True` perfila una corrida de cada uno de los 18 proyectos.** Vale la
pena antes de alinear: en el manifiesto hay **3 proyectos primarios enteros
etiquetados `RNA-Seq`** —`galga PRJEB12164` (27), `maggi PRJNA154615` (21) y
`phypa PRJNA222997` (30)— y **dos son de organismos de entrenamiento**. Si esos
resultan ser mRNA, el modelo se entrena sobre el dato equivocado y no falla
ruidosamente: YASMA anota loci igual, solo que de fragmentos de mRNA.

Son minutos, contra 30-40 h de alineamiento a ciegas.


In [ ]:
TODOS    = True              # True = una corrida de cada proyecto (18)
CORRIDAS = ['SRR23277331']   # si TODOS=False, estas
SPOTS    = 20000

cmd = ['./scripts/fetch_runs.sh', 'perfil']
lotes = [cmd + ['--proyectos', '-n', str(SPOTS)]] if TODOS \
        else [cmd + [r, '-n', str(SPOTS)] for r in CORRIDAS]

for _c in lotes:
    print('$ ' + ' '.join(_c))
    p = subprocess.Popen(_c, cwd=CLON, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    print('exit =', p.wait())


## 2. Encolar las descargas

Recorre **toda** la cola de pendientes. `prefetch` escribe primero en el disco
de la VM, se corre `vdb-validate`, y **recién ahí** se mueve a Drive: escribir
GB directo al FUSE de Drive es lento e inestable, y un `.sra` truncado no falla
ruidosamente —alinea de menos—, así que el que no valida se descarta y nunca
llega al destino.

`HORAS` hace que corte **solo**, antes de que Colab mate la sesión a mitad de
una descarga. Re-ejecutar la celda retoma donde quedó: el estado es qué
archivos existen en Drive, no un contador.

`ORDEN` decide qué se baja primero:

- `entrenamiento` — `gadmo`, `galga` y `maggi` antes que el resto. Son los
  únicos con positivos curados por MirGeneDB, o sea de los que depende que el
  modelo sirva. Es el default.
- `chico` — organismos con menos pendientes primero, para completar organismos
  enteros cuanto antes. Un organismo completo se puede alinear; uno a medias no.
- `alfabetico` — orden fijo, útil si querés que sea predecible.

In [ ]:
ORGANISMO = ''              # '' = todos, o 'prupe', 'gadmo', ...
ORDEN     = 'entrenamiento' # entrenamiento | chico | alfabetico
HORAS     = 3               # corta solo pasadas N horas; None = sin corte
LIMITE    = None            # tope de corridas; None = la cola entera

cmd = ['./scripts/fetch_runs.sh', 'prefetch']
if ORGANISMO:
    cmd.append(ORGANISMO)
cmd += ['--orden', ORDEN]
if HORAS:
    cmd += ['--horas', str(HORAS)]
if LIMITE:
    cmd += ['-n', str(LIMITE)]

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()

## 3. Reparar el ledger

Si una sesión de Colab se muere después de bajar pero antes de que el ledger
llegue a git, el `.sra` queda en Drive y el md5 se pierde con la VM. Esta celda
lo recalcula desde Drive para todo lo que esté bajado y no figure en
`data/sra_md5.tsv`. Solo toca lo que falta, así que correrla de más no hace nada.

`FORMATO` aplica **a las que falten**: una vez movido a Drive el archivo se
llama `<RUN>.sra` tanto si vino normalizado como lite, así que del archivo no se
puede deducir. Las dos de `maggi` (`SRR317135`, `SRR1066790`) son `sralite`.


In [ ]:
ORGANISMO = 'maggi'      # '' = todos
FORMATO   = 'sralite'    # sra | sralite — aplica a las que falten

cmd = ['./scripts/fetch_runs.sh', 'ledger']
if ORGANISMO:
    cmd.append(ORGANISMO)
cmd += ['--formato', FORMATO]

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()


## 4. Guardar el ledger

Los md5 van a git, no solo a Drive: el checksum guardado únicamente al lado del
dato no prueba nada.

El clon **es** un repo git, así que lo que falta commitear es exactamente el
`git diff` del ledger. Con 416 corridas, volcar el archivo entero serían 417
líneas para copiar por las 2 que cambiaron.


In [ ]:
en_repo = set()
_repo_led = CLON / 'data' / 'sra_md5.tsv'
if _repo_led.exists():
    en_repo = set(_repo_led.read_text().strip().split('\n')[1:])

lineas = LEDGER_DRIVE.read_text().strip().split('\n') if LEDGER_DRIVE.exists() else []
faltan = [l for l in lineas[1:] if l not in en_repo]

print(f'ledger en Drive: {max(len(lineas) - 1, 0)} filas')
print(f'ledger en git  : {len(en_repo)} filas')
print()
if faltan:
    print('--- pegale estas filas a data/sra_md5.tsv y commitealas ---')
    print('\n'.join(faltan))
else:
    print('nada que commitear: git ya tiene todo lo que hay en Drive.')


## 5. Repetir

Volvé a correr la celda de la sección 2 hasta que `90_estado.ipynb` no muestre faltantes.
Cada tanda retoma sola, así que alcanza con re-ejecutarla.

**Colab Free no está pensado para trabajo desatendido largo**: el uso sostenido
lleva a throttling. Conviene espaciar las tandas en vez de encadenarlas.

## 6. Diagnóstico de una corrida que falla

Cuando `prefetch` sale con código 0 y aun así no deja un `.sra`, el mensaje de
la celda 2 ya muestra qué quedó en el staging. Esta celda va un paso más atrás:
pregunta al resolver de SRA qué URL devuelve para la corrida, qué dice
`vdb-dump --info` que existe, y corre `prefetch` con **la salida a la vista**.

Con eso se distinguen los dos casos que se confunden: una corrida que solo
existe en formato original (`.fastq.gz`, `.bam`, `.sff`) de una que el resolver
directamente no encuentra. La primera se puede recuperar; la segunda se excluye
del manifiesto y se declara en métodos.


In [ ]:
CORRIDAS = ['SRR317135', 'SRR1066790']   # las que fallan

cmd = ['./scripts/fetch_runs.sh', 'diag'] + CORRIDAS
print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()
